## Étape 4 — Lissage Whittaker-Henderson

Minimise le critère : fidélité aux taux bruts (pondérée par E_x) + régularité (différences d'ordre 2).
Paramètre λ par défaut = 100 (standard actuariel).

In [ ]:
def whittaker_henderson(y, w, lam, d=2):
    n = len(y)
    W = np.diag(w)
    D = np.diff(np.eye(n), n=d, axis=0)
    A = W + lam * D.T @ D
    return np.linalg.solve(A, W @ y)

valid = table['q_x_brut'].notna() & (table['E_x'] > 0) & (table['q_x_brut'] > 0)
n_valid = valid.sum()
y_raw = np.log(table.loc[valid,'q_x_brut'].values.clip(1e-7))
w     = table.loc[valid,'E_x'].values
y_smooth = whittaker_henderson(y_raw, w, LAMBDA_WH)
table.loc[valid,'q_x_lisse'] = np.exp(y_smooth)
t40 = table[table['age'] >= 40].dropna(subset=['q_x_lisse'])
non_mono = (t40['q_x_lisse'].diff().dropna() < 0).sum()
print('Lissage WH applique (lambda={:}) sur {:} ages.'.format(LAMBDA_WH, n_valid))
print('Ages non-monotones apres 40 ans : {:}'.format(non_mono))
if non_mono > 0:
    print('ATTENTION : des inversions existent — augmentez lambda.')
print()
print(table[['age','q_x_brut','q_x_lisse']].dropna().head(15).to_string(index=False))
